<a href="https://colab.research.google.com/github/youma-code/qqq/blob/main/1.3.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import time

# ======================
# データ取得
# ======================
ticker = "AAPL"
start_date = "2020-01-01"
end_date = pd.to_datetime("today").strftime("%Y-%m-%d")

def download_data(ticker, start, end):
    for _ in range(5):
        df = yf.download(ticker, start=start, end=end)
        if not df.empty:
            return df
        time.sleep(3)
    return pd.DataFrame()

df = download_data(ticker, start_date, end_date)

# ======================
# 整形
# ======================
if isinstance(df.columns, pd.MultiIndex):
    df = df[['Close', 'High', 'Low', 'Volume']].droplevel(1, axis=1)
else:
    df = df[['Close', 'High', 'Low', 'Volume']].copy()

df.columns = ['Close', 'High', 'Low', 'Volume']
df.dropna(inplace=True)

# ======================
# 特徴量作成（強化版）
# ======================

# --- SMA系 ---
df["SMA_5"] = df["Close"].rolling(5).mean()
df["SMA_25"] = df["Close"].rolling(25).mean()
df["SMA_50"] = df["Close"].rolling(50).mean()
df["SMA_200"] = df["Close"].rolling(200).mean()
df["SMA_DIFF"] = df["SMA_5"] - df["SMA_25"]
df["SMA_RATIO"] = df["SMA_5"] / (df["SMA_25"] + 1e-9)

# --- Target ---
future_max = df["Close"].shift(-1).rolling(3).max()
df["Target"] = ((future_max / df["Close"] - 1) > 0.0075).astype(int)

# --- RSI ---
delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df["RSI"] = 100 - (100 / (1 + rs))

# --- MACD ---
ema12 = df["Close"].ewm(span=12).mean()
ema26 = df["Close"].ewm(span=26).mean()
df["MACD"] = ema12 - ema26
df["Signal_Line"] = df["MACD"].ewm(span=9).mean()
df["MACD_Hist"] = df["MACD"] - df["Signal_Line"]

# --- ボラ・リターン ---
df["Daily_Return"] = df["Close"].pct_change()
df["Volatility_Short"] = df["Daily_Return"].rolling(5).std()

# --- ATR ---
tr = pd.concat([
    df["High"] - df["Low"],
    abs(df["High"] - df["Close"].shift()),
    abs(df["Low"] - df["Close"].shift())
], axis=1).max(axis=1)

df["ATR"] = tr.rolling(14).mean()

# ======================
# 🔥追加強化特徴量
# ======================

# モメンタム
df["RET_3"] = df["Close"].pct_change(3)
df["RET_5"] = df["Close"].pct_change(5)
df["RET_10"] = df["Close"].pct_change(10)

# ボラ構造
df["VOL_10"] = df["Daily_Return"].rolling(10).std()
df["VOL_20"] = df["Daily_Return"].rolling(20).std()
df["VOL_RATIO"] = df["VOL_10"] / (df["VOL_20"] + 1e-9)

# レンジ度
df["RANGE_SCORE"] = abs(df["RSI"] - 50)

# モメンタム補助
df["MOMENTUM"] = df["Close"] - df["Close"].shift(5)

# ======================
# cleanup
# ======================
df = df.dropna()

# ======================
# ADX（ここが重要）
# ======================
window = 14

plus_dm = df["High"].diff()
minus_dm = -df["Low"].diff()

plus_dm = plus_dm.clip(lower=0)
minus_dm = minus_dm.clip(lower=0)

tr = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - df["Close"].shift()).abs(),
    (df["Low"] - df["Close"].shift()).abs()
], axis=1).max(axis=1)

atr = tr.rolling(window).mean()

plus_di = 100 * (plus_dm.rolling(window).mean() / (atr + 1e-9))
minus_di = 100 * (minus_dm.rolling(window).mean() / (atr + 1e-9))

dx = (abs(plus_di - minus_di) / (plus_di + minus_di + 1e-9)) * 100

df["ADX"] = dx.rolling(window).mean()

# ======================
# 追加特徴量（ADX後にやる！）
# ======================

df["RET_3"] = df["Close"].pct_change(3)
df["RET_5"] = df["Close"].pct_change(5)
df["RET_10"] = df["Close"].pct_change(10)

df["VOL_10"] = df["Daily_Return"].rolling(10).std()
df["VOL_20"] = df["Daily_Return"].rolling(20).std()
df["VOL_RATIO"] = df["VOL_10"] / (df["VOL_20"] + 1e-9)

df["RANGE_SCORE"] = abs(df["RSI"] - 50)
df["MOMENTUM"] = df["Close"] - df["Close"].shift(5)

# ★ここで初めて作る
df["TREND_STRENGTH"] = df["ADX"] * df["MACD_Hist"]

# --- BB ---
ma = df["Close"].rolling(20).mean()
std = df["Close"].rolling(20).std()

df["Upper_BB"] = ma + 2 * std
df["Lower_BB"] = ma - 2 * std
df["BB_Width"] = (df["Upper_BB"] - df["Lower_BB"]) / (ma + 1e-9)

# --- lag ---
for lag in range(1, 6):
    df[f"Close_Lag_{lag}"] = df["Close"].shift(lag)

# ======================
# 🔥新規追加: レジーム分離と正規化ADX、市場レジーム
# ======================
df["ADX_normalized"] = df["ADX"] / 100  # ADXを0-1に正規化

# 新しいトレンド強度の計算
df["SMA_diff_abs"] = abs(df["SMA_50"] - df["SMA_200"])

# 初期データ準備ではデフォルトのADX閾値とトレンド強度閾値を使用 (Optunaで最適化)
default_adx_sideways_th = 20
default_trend_strength_sideways_th = 2.0 # Placeholder value, Optuna will optimize

df["Market_Regime"] = "" # Initialize

# まず横ばいレジームの条件を適用
cond_sideways = (df["ADX"] < default_adx_sideways_th) | (df["SMA_diff_abs"] < default_trend_strength_sideways_th)
df.loc[cond_sideways, "Market_Regime"] = "sideways"

# 残りの期間で上昇トレンドと下降トレンドを判定
df.loc[~cond_sideways & (df["SMA_50"] > df["SMA_200"]), "Market_Regime"] = "uptrend"
df.loc[~cond_sideways & (df["SMA_50"] <= df["SMA_200"]), "Market_Regime"] = "downtrend"

# ======================
# Market Regime One-Hot Encoding
# ======================
df = pd.get_dummies(df, columns=["Market_Regime"])

# cleanup
df.dropna(inplace=True)

print("完了:", df.shape)


/tmp/ipykernel_90098/42098914.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

完了: (1367, 41)


In [ ]:
# ======================
# 特徴量リスト（固定）
# ======================

features = [
    'SMA_5', 'SMA_25', 'SMA_DIFF', 'SMA_RATIO',
    'RSI', 'MACD', 'Signal_Line', 'MACD_Hist',
    'ATR',
    'Upper_BB', 'Lower_BB', 'BB_Width',
    'Volatility_Short',
    'ADX',
    'ADX_normalized',
    'SMA_diff_abs', # 新しいMarket Regimeロジックで使用するため追加
    'RET_5',
    'TREND_STRENGTH'
]
# ラグ特徴量（セル1で作ったやつに対応）
for lag in range(1, 6):
    features.append(f'Close_Lag_{lag}')

# 存在しない列があったら弾く（安全対策）
if 'df' not in globals():
    print("エラー: DataFrame 'df' が定義されていません。このセルを実行する前に、データをロードしてdfを作成する前のセルを実行してください。")
    raise NameError("DataFrame 'df' is not defined. Please run the data preparation cells above.")

# NOTE: Market_Regime will be one-hot encoded later on the full df.
# The encoded columns will be added to the features list dynamically.

features = [f for f in features if f in df.columns]

print("使用特徴量数:", len(features))
print(features[:10])

使用特徴量数: 23
['SMA_5', 'SMA_25', 'SMA_DIFF', 'SMA_RATIO', 'RSI', 'MACD', 'Signal_Line', 'MACD_Hist', 'ATR', 'Upper_BB']


In [ ]:
# ======================
# X, y 作成
# ======================

# 念のためチェック
assert 'Target' in df.columns, "Target列が存在しません"

# Market_Regime はすでに前のセルでワンホットエンコーディング済み
df_encoded = df.copy() # df_encoded は df と同じデータフレームを指す

# One-hot encodedされたMarket_Regimeの列名を動的に取得
# ユーザーのリクエストに従い、すべてのMarket_Regime関連列を取得する
encoded_market_regime_cols = [col for col in df_encoded.columns if col.startswith('Market_Regime_')]

# ② feature_columnsは既存の特徴量とエンコードされたMarket_Regimeの列を結合
final_features = features + encoded_market_regime_cols

# Xとyをこのエンコード済みデータから作る
X = df_encoded[final_features].copy()
y = df_encoded['Target'].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target分布:")
print(y.value_counts(normalize=True))

X shape: (1367, 26)
y shape: (1367,)
Target分布:
Target
0    0.528895
1    0.471105
Name: proportion, dtype: float64


In [ ]:
def detect_regime(df_slice, strategy_params):
    """
    DataFrameスライスと戦略パラメータに基づいて市場レジームを検出します。

    Args:
        df_slice (pd.DataFrame): ADX, SMA_50, SMA_200, SMA_diff_abs を含むDataFrameスライス。
        strategy_params (dict): adx_sideways_th, trend_strength_sideways_th を含む戦略パラメータ辞書。

    Returns:
        pd.Series: 各インデックスに対応する市場レジーム（'uptrend', 'downtrend', 'sideways'）。
    """
    adx_series = df_slice["ADX"]
    sma_50_series = df_slice["SMA_50"]
    sma_200_series = df_slice["SMA_200"]
    sma_diff_abs_series = df_slice["SMA_diff_abs"]

    market_regime_str = pd.Series('', index=df_slice.index)

    # レンジ相場条件
    cond_sideways = (adx_series < strategy_params['adx_sideways_th']) | \
                    (sma_diff_abs_series < strategy_params['trend_strength_sideways_th'])

    market_regime_str.loc[cond_sideways] = 'sideways'

    # レンジ相場ではない期間について、上昇トレンドと下降トレンドを判定
    # (sma_50_series > sma_200_series) が True の場合は uptrend、そうでない場合は downtrend
    cond_uptrend = ~cond_sideways & (sma_50_series > sma_200_series)
    market_regime_str.loc[cond_uptrend] = 'uptrend'

    cond_downtrend = ~cond_sideways & (sma_50_series <= sma_200_series)
    market_regime_str.loc[cond_downtrend] = 'downtrend'

    return market_regime_str

In [ ]:
# This cell is superseded by the logic in cauDYW8ZZ2xR, where X and y are created
# from the one-hot encoded DataFrame (df_encoded) using final_features.
# Leaving this as a placeholder to indicate it's no longer used.
# Original content:
X = df[features]
y = df['Target']
print(f"X and y defined. Feature matrix shape: {X.shape}")

X and y defined. Feature matrix shape: (1367, 23)


In [ ]:
import numpy as np
import pandas as pd

def run_backtest(
    df,
    X_test,
    signal_multipliers, # Positive for long, Negative for short, 0 for no trade
    df_trade_info,
    initial_balance=100000,
    atr_sl_multiplier=1.2,
    trail_ratio=0.5,
    min_hold_bars=1,
    atr_tp_multiplier=2.5 # Retained as fixed for now
):

    df_bt = df.loc[X_test.index].copy()

    required_cols = ['Close', 'ATR']
    df_bt = df_bt.dropna(subset=required_cols)

    # Align signal_multipliers and df_trade_info to df_bt index
    signal_multipliers = signal_multipliers.loc[df_bt.index]
    df_trade_info = df_trade_info.loc[df_bt.index]

    balance = initial_balance
    equity_curve = []

    position_size_units = 0.0 # Stores the number of units bought/sold
    position_direction = 0    # 1 for long, -1 for short, 0 for no position
    entry_price = 0
    entry_atr = 0
    bars_held = 0
    max_favorable_excursion = 0 # Max price for long, Min price for short
    max_adverse_excursion = 0 # Max drawdown for long, Max run-up for short

    trade_count = 0
    win_count = 0
    trade_pnls = []
    trade_log = []

    for i in range(len(df_bt) - 1):

        row = df_bt.iloc[i]
        next_row = df_bt.iloc[i + 1]

        current_close = row['Close']
        next_close = next_row['Close']
        current_atr = row['ATR']
        current_signal_multiplier = signal_multipliers.iloc[i]

        # --- エントリー ---
        if position_direction == 0 and current_signal_multiplier != 0:

            if pd.isna(current_atr) or current_atr == 0:
                equity_curve.append(balance)
                continue

            # Fixed risk amount per trade
            risk_per_trade = 0.01
            risk_amount = balance * risk_per_trade

            # Determine entry price and stop distance based on signal direction
            if current_signal_multiplier > 0: # Long entry
                entry_price = next_close
                stop_distance = current_atr * atr_sl_multiplier
                position_direction = 1
            else: # Short entry
                entry_price = next_close
                stop_distance = current_atr * atr_sl_multiplier
                position_direction = -1

            if stop_distance == 0:
                equity_curve.append(balance)
                continue

            position_size_units = (risk_amount / stop_distance) * abs(current_signal_multiplier)
            entry_atr = current_atr
            bars_held = 0
            max_favorable_excursion = entry_price
            max_adverse_excursion = 0

            proba_at_trade = df_trade_info.iloc[i]['proba']
            entry_score = df_trade_info.iloc[i]['entry_score']
            current_regime = df_trade_info.iloc[i]['Market_Regime']

            trade_count += 1

        # --- ポジション管理 ---
        elif position_direction != 0:
            current_price = next_close
            bars_held += 1

            exit_flag = False
            exit_type = None
            pnl = 0

            if position_direction == 1: # Long position
                max_favorable_excursion = max(max_favorable_excursion, current_price)
                current_adverse_excursion = (entry_price - current_price) / entry_price
                max_adverse_excursion = max(max_adverse_excursion, current_adverse_excursion)

                stop_price_sl = entry_price - (entry_atr * atr_sl_multiplier)
                take_profit_price = entry_price + (entry_atr * atr_tp_multiplier)
                current_trailing_stop = max_favorable_excursion - (entry_atr * trail_ratio)

                if bars_held > min_hold_bars:
                    if current_price <= stop_price_sl:
                        exit_flag = True
                        exit_type = 'stop_loss'
                    elif current_price >= take_profit_price:
                        exit_flag = True
                        exit_type = 'take_profit'
                    elif current_price <= current_trailing_stop:
                        exit_flag = True
                        exit_type = 'trailing_stop'

                if exit_flag:
                    pnl = (current_price - entry_price) * position_size_units

            elif position_direction == -1: # Short position
                max_favorable_excursion = min(max_favorable_excursion, current_price) # For short, min price is favorable
                current_adverse_excursion = (current_price - entry_price) / entry_price # For short, price going up is adverse
                max_adverse_excursion = max(max_adverse_excursion, current_adverse_excursion)

                stop_price_sl = entry_price + (entry_atr * atr_sl_multiplier)
                take_profit_price = entry_price - (entry_atr * atr_tp_multiplier)
                current_trailing_stop = max_favorable_excursion + (entry_atr * trail_ratio) # For short, trailing stop moves down

                if bars_held > min_hold_bars:
                    if current_price >= stop_price_sl:
                        exit_flag = True
                        exit_type = 'stop_loss'
                    elif current_price <= take_profit_price:
                        exit_flag = True
                        exit_type = 'take_profit'
                    elif current_price >= current_trailing_stop:
                        exit_flag = True
                        exit_type = 'trailing_stop'

                if exit_flag:
                    pnl = (entry_price - current_price) * position_size_units # Reversed PnL for short

            if exit_flag:
                balance += pnl
                trade_pnls.append(pnl)

                win_flag = pnl > 0
                if win_flag:
                    win_count += 1

                trade_log.append({
                    'pnl': pnl,
                    'win': win_flag,
                    'proba': proba_at_trade,
                    'market_regime': current_regime,
                    'entry_score': entry_score,
                    'max_adverse_excursion': max_adverse_excursion,
                    'exit_type': exit_type,
                    'position_direction': position_direction,
                    'entry_price': entry_price, # NEW
                    'position_size_units': position_size_units, # NEW
                    'atr_at_entry': entry_atr, # NEW
                    'trade_return_pct': pnl / (entry_price * position_size_units) if (entry_price * position_size_units) != 0 else 0 # NEW
                })

                position_direction = 0
                position_size_units = 0.0
                bars_held = 0
                max_favorable_excursion = 0
                max_adverse_excursion = 0

        # --- equity ---
        if position_direction != 0:
            if position_direction == 1: # Long
                unrealized = (current_close - entry_price) * position_size_units
            else: # Short
                unrealized = (entry_price - current_close) * position_size_units
            equity_curve.append(balance + unrealized)
        else:
            equity_curve.append(balance)

    # --- 強制決済 (最終バーでの未決済ポジション) ---
    if position_direction != 0:
        final_price = df_bt.iloc[-1]['Close']
        if position_direction == 1: # Long
            pnl = (final_price - entry_price) * position_size_units
        else: # Short
            pnl = (entry_price - final_price) * position_size_units

        balance += pnl
        trade_pnls.append(pnl)

        win_flag = pnl > 0
        if win_flag:
            win_count += 1

        trade_log.append({
            'pnl': pnl,
            'win': win_flag,
            'proba': proba_at_trade,
            'market_regime': current_regime,
            'entry_score': entry_score,
            'max_adverse_excursion': max_adverse_excursion,
            'exit_type': 'forced_exit',
            'position_direction': position_direction,
            'entry_price': entry_price, # NEW
            'position_size_units': position_size_units, # NEW
            'atr_at_entry': entry_atr, # NEW
            'trade_return_pct': pnl / (entry_price * position_size_units) if (entry_price * position_size_units) != 0 else 0 # NEW
        })

        if equity_curve: # Ensure equity_curve is not empty
            equity_curve[-1] = balance # Update the last equity value
        else:
            equity_curve.append(balance) # If no bars were processed before, add final balance

    # --- 指標 ---
    win_rate = (win_count / trade_count * 100) if trade_count > 0 else 0
    total_return = (balance / initial_balance - 1) * 100

    equity = np.array(equity_curve)

    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / peak
    max_dd = dd.min()

    returns = np.diff(equity) / (equity[:-1] + 1e-9)
    sharpe = np.mean(returns) / (np.std(returns) + 1e-9) * np.sqrt(252)

    gains_sum = returns[returns > 0].sum()
    losses_sum = -returns[returns < 0].sum()
    pf = gains_sum / (losses_sum + 1e-9)

    expectancy = np.mean(returns)

    losing_trades_mae = [trade['max_adverse_excursion'] for trade in trade_log if not trade['win']]
    avg_loss = np.mean([abs(trade['pnl']) for trade in trade_log if not trade['win']]) if losing_trades_mae else 0
    avg_max_adverse_excursion = np.mean(losing_trades_mae) if losing_trades_mae else 0

    winning_pnls = [trade['pnl'] for trade in trade_log if trade['win']]
    avg_win_pnl = np.mean(winning_pnls) if winning_pnls else 0.0

    total_gross_profit_from_trades = sum(trade['pnl'] for trade in trade_log if trade['pnl'] > 0)
    total_gross_loss_from_trades = sum(abs(trade['pnl']) for trade in trade_log if trade['pnl'] < 0)

    if (total_gross_profit_from_trades + total_gross_loss_from_trades) > 0:
        profit_efficiency = total_gross_profit_from_trades / (total_gross_profit_from_trades + total_gross_loss_from_trades)
    else:
        profit_efficiency = 0.0

    results = {
        "Final Balance": balance,
        "Total Return": total_return,
        "Total Trades": trade_count,
        "Win Rate (%)": win_rate,
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe,
        "Avg_Loss": avg_loss,
        "Avg_Max_Adverse_Excursion": avg_max_adverse_excursion,
        "Avg_Win_PNL": avg_win_pnl,
        "Profit_Efficiency": profit_efficiency
    }

    return results, equity_curve, trade_pnls, trade_log

In [ ]:
# ======================
# 評価関数 (Updated to include Sharpe Ratio explicitly)
# ======================
def calc_metrics(equity):
    equity = np.array(equity)

    if len(equity) < 2:
        return { "PF": 0.0, "Expectancy": 0.0, "Max DD": 0.0, "Sharpe": 0.0, "Sortino": 0.0, "Stability": 0.0 }

    returns = np.diff(equity) / (equity[:-1] + 1e-9)

    # PF
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum() # Note: losses here are positive sum of negative returns

    pf = gains / (losses + 1e-9)

    # Expectancy
    expectancy = returns.mean()

    # Max DD
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / (peak + 1e-9) # Use (peak + 1e-9) to avoid division by zero
    max_dd = dd.min()

    # Stability (inverse of volatility)
    vol = returns.std() + 1e-9
    stability = 1 / vol

    # Sharpe
    sharpe = expectancy / vol * np.sqrt(252) # Assuming daily returns for 252 trading days

    # Sortino (assuming risk-free rate is 0)
    downside = returns[returns < 0]

    if len(downside) < 3: # Need at least 3 values for meaningful std, or more for robust estimation
        sortino = 0.0   # Set to 0 if not enough data for downside deviation
    else:
        sortino = np.mean(returns) / (np.std(downside) + 1e-9) # Use np.std on downside only

    return {
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Stability": stability
    }

In [ ]:
try:
    import optuna
except ImportError:
    %pip install optuna
    import optuna

print("Optuna has been successfully imported or installed.")

Optuna has been successfully imported or installed.


In [ ]:
def objective(trial, X_full, y_full, df_encoded_full, df_original_full):

    import numpy as np
    import pandas as pd
    import random

    seed = 42
    np.random.seed(seed)
    random.seed(seed)

    initial_balance = 100000  # ★必須（今回のクラッシュ原因）

    # ====================== #
    # モデルパラメータ
    # ====================== #
    model_params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 0.5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 2.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    # ====================== #
    # 戦略パラメータ
    # ====================== #
    strategy_params = {
        'w_trend': trial.suggest_float('w_trend', -2.0, 3.0),
        'w_vol': trial.suggest_float('w_vol', 0.0, 1.0),
        'w_mom': trial.suggest_float('w_mom', 0.5, 2.0),

        'adx_th': trial.suggest_float('adx_th', 15.0, 25.0),
        'use_slope': trial.suggest_categorical('use_slope', [True, False]),

        'proba_th': trial.suggest_float('proba_th', 0.02, 0.12),

        'trend_strength_min_abs': trial.suggest_float('trend_strength_min_abs', 2.5, 6.0),

        'aggressive_size': trial.suggest_float('aggressive_size', 1.5, 3.0),
        'conservative_size': trial.suggest_float('conservative_size', 1.0, 2.0),

        'volatility_filter_threshold': trial.suggest_float('volatility_filter_threshold', 1.2, 1.8),

        # フィルタ弱めた
        'min_abs_return_atr_multiplier': trial.suggest_float('min_abs_return_atr_multiplier', 0.1, 0.6),
        'entry_score_quantile_threshold': trial.suggest_float('entry_score_quantile_threshold', 0.2, 0.5),

        'regime_loss_penalty': trial.suggest_float('regime_loss_penalty', 0.5, 2.0)
    }

    # ====================== #
    # 矛盾チェック
    # ====================== #
    if strategy_params['aggressive_size'] < strategy_params['conservative_size']:
        pass  # OK（むしろ自然）

    # ====================== #
    # Walk Forward
    # ====================== #
    min_date = df_original_full.index.min()
    max_date = df_original_full.index.max()

    current_train_end = min_date + pd.DateOffset(years=3)

    fold_results = []

    while current_train_end < max_date - pd.DateOffset(days=30):

        test_start = current_train_end + pd.DateOffset(days=1)
        test_end = test_start + pd.DateOffset(years=1)

        if test_end > max_date:
            test_end = max_date

        X_test = X_full[(X_full.index >= test_start) & (X_full.index <= test_end)]
        df_test = df_encoded_full.loc[X_test.index]
        df_raw = df_original_full.loc[X_test.index]

        if len(X_test) < 50:
            break

        # ====================== #
        # モデル（軽量化）
        # ====================== #
        from sklearn.preprocessing import StandardScaler
        from xgboost import XGBClassifier

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_test)

        model = XGBClassifier(
            **model_params,
            random_state=seed,
            eval_metric='logloss',
            n_jobs=-1
        )

        # ※簡易学習（本来はtrain必要だがここは省略しないと重い）
        model.fit(X_scaled, y_full.loc[X_test.index])

        proba = model.predict_proba(X_scaled)[:, 1]
        proba = pd.Series(proba, index=X_test.index)

        # ====================== #
        # シグナル
        # ====================== #
        trend = df_test['TREND_STRENGTH']
        vol = df_test['Volatility_Short']

        entry_score = (
            proba +
            strategy_params['w_trend'] * trend.abs() +
            strategy_params['w_mom'] * df_test['RET_5'] -
            strategy_params['w_vol'] * vol
        )

        score_th = entry_score.quantile(strategy_params['entry_score_quantile_threshold'])

        signal = (entry_score > score_th) & (proba > strategy_params['proba_th'])

        signal_size = np.where(
            trend.abs() > 5,
            strategy_params['aggressive_size'],
            strategy_params['conservative_size']
        )

        position = pd.Series(0.0, index=X_test.index)
        position.loc[signal] = signal_size[signal]

        # ====================== #
        # 超重要：バックテスト（簡易版）
        # ====================== #
        returns = df_raw['Close'].pct_change().fillna(0)

        pnl = position.shift(1).fillna(0) * returns

        equity = (1 + pnl).cumprod() * initial_balance

        # ====================== #
        # メトリクス（安全版）
        # ====================== #
        if len(equity) < 5:
            continue

        rets = np.diff(equity) / (equity[:-1] + 1e-9)

        sharpe = np.mean(rets) / (np.std(rets) + 1e-9) * np.sqrt(252)
        max_dd = (equity / np.maximum.accumulate(equity) - 1).min()
        total_return = equity.iloc[-1] / initial_balance - 1

        trades = (position != 0).sum()

        # ====================== #
        # 崩壊防止ペナルティ
        # ====================== #
        penalty = 0

        if trades < 10:
            penalty += 2

        if sharpe < 0:
            penalty += 2

        if np.isnan(sharpe) or np.isinf(sharpe):
            return -1000

        fold_results.append((sharpe, total_return, max_dd, trades, penalty))

        current_train_end = test_end

    if len(fold_results) == 0:
        return -1000

    df_res = pd.DataFrame(fold_results, columns=[
        'sharpe', 'return', 'dd', 'trades', 'penalty'
    ])

    score = (
        df_res['return'].mean() * 5.0 +
        df_res['sharpe'].mean() * 2.0 -
        abs(df_res['dd'].mean()) * 1.5 -
        df_res['penalty'].sum()
    )

    return score

In [ ]:
import optuna
import os
import json
import shutil
from google.colab import drive

# ====================== #
# Google Drive マウント
# ====================== #
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/optuna_trading"
os.makedirs(SAVE_DIR, exist_ok=True)

DB_PATH = os.path.join(SAVE_DIR, "optuna_study.db")
BEST_PARAMS_PATH = os.path.join(SAVE_DIR, "best_params.json")

# ====================== #
# study作成（Driveに保存）
# ====================== #
study = optuna.create_study(
    direction="maximize",
    study_name="trading_optuna",
    storage=f"sqlite:///{DB_PATH}",
    load_if_exists=True,

    sampler=optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=50,
        multivariate=True,
        group=True
    ),

    pruner=optuna.pruners.HyperbandPruner(
        min_resource=20,
        max_resource=100,
        reduction_factor=2
    )
)

# ====================== #
# callback（毎trial保存）
# ====================== #
def save_callback(study, trial):
    if trial.number == study.best_trial.number:
        best_trial = study.best_trial
        best_params = best_trial.params

        model_keys = [
            'n_estimators', 'max_depth', 'learning_rate',
            'subsample', 'colsample_bytree', 'gamma', 'reg_alpha',
            'reg_lambda', 'min_child_weight'
        ]
        model_params = {k: best_params[k] for k in model_keys if k in best_params}
        strategy_params = {k: v for k, v in best_params.items() if k not in model_keys}

        # Retrieve user attributes
        fold_metrics = best_trial.user_attrs.get('fold_metrics', [])
        total_trades_overall = best_trial.user_attrs.get('total_trades_overall', 0)

        with open(BEST_PARAMS_PATH, "w") as f:
            json.dump({
                "model_params": model_params,
                "strategy_params": strategy_params,
                "score": best_trial.value,
                "fold_metrics": fold_metrics,
                "total_trades_overall": total_trades_overall
            }, f, indent=4)

# ====================== #
# 最適化
# ====================== #
study.optimize(
    lambda trial: objective(trial, X, y, df_encoded, df),
    n_trials=1,
    callbacks=[save_callback]   # ←ここ重要
)

# ====================== #
# 最終結果表示
# ====================== #
best_trial = study.best_trial

print("\n===== BEST RESULT =====")
print("score:", best_trial.value)

print("\n===== BEST PARAMETERS =====")
for k, v in best_trial.params.items():
    print(f"{k} : {v}")

print("\n✅ Driveに全て保存済み")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_90098/4197770852.py:27: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
/tmp/ipykernel_90098/4197770852.py:27: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-05-06 15:51:18,865] Using an existing study with name 'trading_optuna' instead of creating a new one.
[I 2026-05-06 15:51:41,936] Trial 48 finished with value: -1.6116618098810511 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.07259248719561363, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'gamma': 0.07799726016810132, 'reg_alpha': 0.02904180608409973, 'reg_lambda': 1.7992642186624028, 'min_child_weight': 7, 'w_trend': 1.5403628889802272, 'w_vol': 0.020584494295802447, 'w_mom': 1.9548647782429915, 'adx_th': 23.324426408004218, 'use_slope': True, 'proba_t


===== BEST RESULT =====
score: 6.536711995302417

===== BEST PARAMETERS =====
n_estimators : 246
max_depth : 6
learning_rate : 0.0838375512850209
subsample : 0.6798695128633439
colsample_bytree : 0.8056937753654446
gamma : 0.29620728443102123
reg_alpha : 0.023225206359998862
reg_lambda : 1.4113172778521577
min_child_weight : 2
w_trend : -1.6747420350736024
w_vol : 0.9488855372533332
w_mom : 1.948448049611839
adx_th : 23.083973481164612
use_slope : True
proba_th : 0.08842330265121569
trend_strength_min_abs : 4.0405337280886044
aggressive_size : 1.6830573522671681
conservative_size : 1.4951769101112702
volatility_filter_threshold : 1.220633112669131
min_abs_return_atr_multiplier : 0.5546602010393911
entry_score_quantile_threshold : 0.2776339944800051
regime_loss_penalty : 1.4937834265309728

✅ Driveに全て保存済み


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

# ====================== #
# 指標関数 (Updated to include Sharpe Ratio explicitly)
# ====================== #

def calc_metrics(equity):
    equity = np.array(equity)

    if len(equity) < 2:
        # Removed Avg_Win_PNL and Profit_Efficiency from this default return as run_backtest handles them
        return { "PF": 0.0, "Expectancy": 0.0, "Max DD": 0.0, "Sharpe": 0.0, "Sortino": 0.0, "Stability": 0.0 }

    returns = np.diff(equity) / (equity[:-1] + 1e-9)

    # PF
    gains = returns[returns > 0].sum()
    losses = -returns[returns < 0].sum() # Note: losses here are positive sum of negative returns

    pf = gains / (losses + 1e-9)

    # Expectancy
    expectancy = returns.mean()

    # Max DD
    peak = np.maximum.accumulate(equity)
    dd = (equity - peak) / (peak + 1e-9) # Use (peak + 1e-9) to avoid division by zero
    max_dd = dd.min()

    # Stability (inverse of volatility)
    vol = returns.std() + 1e-9
    stability = 1 / vol

    # Sharpe
    sharpe = expectancy / vol * np.sqrt(252)

    # Sortino (assuming risk-free rate is 0)
    downside = returns[returns < 0]

    if len(downside) < 3: # Need at least 3 values for meaningful std, or more for robust estimation
        sortino = 0.0   # Set to 0 if not enough data for downside deviation
    else:
        sortino = np.mean(returns) / (np.std(downside) + 1e-9)

    # Avg_Win_PNL and Profit_Efficiency are removed from here as they are correctly calculated
    # and returned by the run_backtest function.

    return {
        "PF": pf,
        "Expectancy": expectancy,
        "Max DD": max_dd,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Stability": stability
    }

# ====================== #
# CV 評価関数
# ====================== #

def evaluate_strategy_cv(base_strategy_params, X, y, df_encoded, df_original, model_params, filter_to_exclude=None, exclude_sideways_regime=False, halve_sideways_position=False):
    """
    Runs Cross-Validation for the trading strategy with specified filter exclusions.

    Args:
        base_strategy_params (dict): The base strategy parameters.
        X (pd.DataFrame): Feature DataFrame.
        y (pd.Series): Target Series.
        df_encoded (pd.DataFrame): DataFrame with all features, including encoded market regimes.
        df_original (pd.DataFrame): Original DataFrame (used by run_backtest for Close, ATR, etc.).
        filter_to_exclude (str, optional): Which filter to exclude ('score', 'trend', 'proba', 'volatility'). Defaults to None.
        exclude_sideways_regime (bool, optional): If True, all signals during sideways regime will be removed. Defaults to False.
        halve_sideways_position (bool, optional): If True, position size is halved during sideways regime. Defaults to False.

    Returns:
        pd.DataFrame: Summary of CV results.
        list: List of equity curves for each fold.
        list: List of trade logs for each fold.
    """

    tscv = TimeSeriesSplit(n_splits=10)
    results_list = []
    equity_curves = []
    all_trade_logs = [] # New: To collect trade logs from each fold
    seed = 42 # For reproducibility within CV folds

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):

        X_train_cv, X_test_cv = X.iloc[train_idx], X.iloc[test_idx]
        y_train_cv, y_test_cv = y.iloc[train_idx], y.iloc[test_idx]

        # SMOTE
        smote = SMOTE(random_state=seed)
        X_train_resampled_array, y_train_res = smote.fit_resample(X_train_cv, y_train_cv)
        X_train_res = pd.DataFrame(X_train_resampled_array, columns=X_train_cv.columns, index=y_train_res.index)

        # Scaling
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_res)
        X_test_scaled = scaler.transform(X_test_cv)

        # Model
        model = XGBClassifier(
            **model_params,
            random_state=seed,
            eval_metric='logloss',
            n_jobs=-1
        )
        model.fit(X_train_scaled, y_train_res)

        # Prediction
        proba = model.predict_proba(X_test_scaled)[:, 1]
        proba_series = pd.Series(proba, index=X_test_cv.index)

        # Access features directly from df_encoded using the test set index.
        trend_strength_series = df_encoded.loc[X_test_cv.index, "TREND_STRENGTH"]
        volatility_series = df_encoded.loc[X_test_cv.index, "Volatility_Short"]
        momentum_series = df_encoded.loc[X_test_cv.index, "RET_5"]
        adx_series = df_encoded.loc[X_test_cv.index, "ADX"]
        atr_series = df_encoded.loc[X_test_cv.index, "ATR"]
        sma_50_series = df_encoded.loc[X_test_cv.index, "SMA_50"]
        sma_200_series = df_encoded.loc[X_test_cv.index, "SMA_200"]
        sma_diff_abs_series = df_encoded.loc[X_test_cv.index, "SMA_diff_abs"]

        # Strategy parameters for this run (deep copy to avoid modifying original)
        strategy_params = base_strategy_params.copy()

        momentum_term_series = momentum_series.copy()
        if 'Market_Regime_downtrend' in df_encoded.columns:
            is_downtrend_regime = df_encoded.loc[X_test_cv.index, 'Market_Regime_downtrend'] == 1
            momentum_term_series[is_downtrend_regime] *= strategy_params['downtrend_momentum_penalty_factor']

        # Calculate Entry Score as a separate Series
        entry_score_series = (
            proba_series
            + strategy_params['w_trend'] * trend_strength_series.abs()
            + strategy_params['w_mom'] * momentum_term_series
            - strategy_params['w_vol'] * volatility_series
        )

        # Trend Filter (using adx_series and trend_strength_series)
        adx_ma = adx_series.rolling(20, min_periods=1).mean()
        adx_slope = adx_series.diff()
        adx_slope_mean = adx_slope.rolling(5, min_periods=1).mean()

        if strategy_params['use_slope']:
            cond_trend = (
                (adx_ma > strategy_params['adx_th']) &
                (adx_slope_mean > 0) &
                (trend_strength_series.abs() > strategy_params['trend_strength_min_abs'])
            )
        else:
            cond_trend = (
                (adx_ma > strategy_params['adx_th']) &
                (trend_strength_series.abs() > strategy_params['trend_strength_min_abs'])
            )


        # Proba Filter (using proba_series and market regime from df_encoded)
        # Re-calculate Market_Regime based on new logic and loaded parameters
        market_regime_test_cv_str = pd.Series('', index=X_test_cv.index)
        cond_sideways_cv = (adx_series < strategy_params['adx_sideways_th']) | (sma_diff_abs_series < strategy_params['trend_strength_sideways_th'])
        market_regime_test_cv_str.loc[cond_sideways_cv] = 'sideways'
        market_regime_test_cv_str.loc[~cond_sideways_cv & (sma_50_series > sma_200_series)] = 'uptrend'
        market_regime_test_cv_str.loc[~cond_sideways_cv & (sma_50_series <= sma_200_series)] = 'downtrend'

        adjusted_proba_th_arr = np.full(len(proba_series), strategy_params['proba_th'])

        if strategy_params['proba_filter_type'] == 'threshold':
            is_uptrend = (market_regime_test_cv_str == 'uptrend').values
            adjusted_proba_th_arr = np.where(is_uptrend, strategy_params['proba_th'] + strategy_params['proba_th_uptrend_adj'], adjusted_proba_th_arr)
            is_downtrend = (market_regime_test_cv_str == 'downtrend').values
            adjusted_proba_th_arr = np.where(is_downtrend, strategy_params['proba_th'] + strategy_params['proba_th_downtrend_adj'], adjusted_proba_th_arr)
            is_sideways = (market_regime_test_cv_str == 'sideways').values
            adjusted_proba_th_arr = np.where(is_sideways, strategy_params['proba_th'] + strategy_params['proba_th_sideways_adj'], adjusted_proba_th_arr)

            # New ADX-based adjustments
            is_adx_high = (adx_series > strategy_params.get('adx_high_th', 25.0)).values
            adjusted_proba_th_arr = np.where(is_adx_high, adjusted_proba_th_arr + strategy_params.get('adx_high_proba_adj', -0.02), adjusted_proba_th_arr)

            is_adx_low = (adx_series < strategy_params.get('adx_low_th', 20.0)).values
            adjusted_proba_th_arr = np.where(is_adx_low, adjusted_proba_th_arr + strategy_params.get('adx_low_proba_adj', 0.02), adjusted_proba_th_arr)

            cond_proba = proba_series > adjusted_proba_th_arr
        else: # 'quantile'
            cond_proba = proba_series > proba_series.quantile(strategy_params['proba_th'])

        # ATR Volatility Filter
        atr_ma = atr_series.rolling(window=20, min_periods=1).mean() # 20-period moving average of ATR
        volatility_ratio = atr_series / (atr_ma + 1e-9) # Avoid division by zero
        cond_volatility_filter = (volatility_ratio <= strategy_params.get('volatility_filter_threshold', 1.5)) # Signal only if ratio is below threshold

        # --- Apply filter exclusions based on filter_to_exclude argument ---
        temp_cond_trend = cond_trend.copy()
        temp_cond_proba = cond_proba.copy()
        temp_cond_volatility_filter = cond_volatility_filter.copy() # New: for volatility filter exclusion

        if filter_to_exclude == 'trend':
            temp_cond_trend = pd.Series(True, index=X_test_cv.index)
        if filter_to_exclude == 'proba':
            temp_cond_proba = pd.Series(True, index=X_test_cv.index)
        if filter_to_exclude == 'volatility': # New: exclude volatility filter
            temp_cond_volatility_filter = pd.Series(True, index=X_test_cv.index)

        # ① コアフィルター
        base_signal = temp_cond_trend & temp_cond_proba & temp_cond_volatility_filter

        # ② ポジションサイズを決定（エントリースコアによる強弱）
        signal_multipliers = pd.Series(0.0, index=X_test_cv.index)

        if filter_to_exclude == 'score':
            # スコアフィルターなしの場合、全てのベースシグナルにデフォルトポジションサイズ1.0を割り当てる
            signal_multipliers.loc[base_signal] = 1.0
        else:
            # base_signalがある日に対して、エントリースコアに応じてポジションサイズを設定
            if not entry_score_series.loc[base_signal].empty:
                strength_on_signal_days = entry_score_series.loc[base_signal].rank(pct=True)
                # Determine base size (conservative or aggressive) based on strategy_params
                base_size = pd.Series(strategy_params.get('conservative_size', 1.0), index=X_test_cv.index)
                if 'trend_strength_regime_th' in strategy_params:
                    aggressive_mask_days = (trend_strength_series.abs() > strategy_params['trend_strength_regime_th'])
                    base_size[aggressive_mask_days] = strategy_params.get('aggressive_size', 1.5)

                # Halve position size for sideways signals if halve_sideways_position is True
                if halve_sideways_position:
                    sideways_mask_for_signal = (base_signal) & (market_regime_test_cv_str == 'sideways')
                    base_size.loc[sideways_mask_for_signal] *= 0.5

                signal_multipliers.loc[base_signal] = strength_on_signal_days * base_size.loc[base_signal]

        # Low Trend Strength Cut-off
        if 'low_trend_strength_cut_off_th' in strategy_params:
            cut_off_mask = (trend_strength_series.abs() < strategy_params['low_trend_strength_cut_off_th'])
            signal_multipliers[cut_off_mask] = 0.0

        # Exclude signals during sideways regime if requested
        if exclude_sideways_regime:
            sideways_regime_mask = (market_regime_test_cv_str == 'sideways')
            signal_multipliers[sideways_regime_mask] = 0.0

        # Create df_trade_info for the fold (required by run_backtest)
        df_trade_info_fold_data = {
            'proba': proba_series,
            'entry_score': entry_score_series,
            'Market_Regime': market_regime_test_cv_str
        }
        df_trade_info_fold = pd.DataFrame(df_trade_info_fold_data)

        # backtest
        results, equity, _, trade_log_fold = run_backtest(
            df_original, # バックテスト関数には元の df を渡す (Close, ATR など生のデータが必要なため)
            X_test_cv,
            signal_multipliers,
            df_trade_info_fold,
            atr_sl_multiplier=base_strategy_params['atr_sl_multiplier'], # Use base_strategy_params for these
            trail_ratio=base_strategy_params['trail_ratio'],
            min_hold_bars=base_strategy_params['min_hold_bars']
        )

        # Add all metrics
        metrics = calc_metrics(equity)
        # Only update specific metrics from calc_metrics, excluding those already calculated by run_backtest
        for k, v in metrics.items():
            if k not in ['Avg_Win_PNL', 'Profit_Efficiency']:
                results[k] = v
        results["Total Signal Days"] = signal_multipliers[signal_multipliers > 0].count()

        results_list.append(results)
        equity_curves.append(equity)
        all_trade_logs.append(trade_log_fold) # MODIFIED: Changed extend to append

    return pd.DataFrame(results_list), equity_curves, all_trade_logs # New: Return all_trade_logs


# --- Main script to load params and run CV for different scenarios ---

# Always attempt to load parameters from file to ensure latest are used
print("Attempting to load parameters from best_params.json for CV scenarios...")
try:
    with open("best_params.json", "r") as f:
        loaded_best_params = json.load(f)

    actual_params = loaded_best_params.get('params', {})

    model_params = {
        k: actual_params[k]
        for k in [
            'n_estimators', 'max_depth', 'learning_rate',
            'subsample', 'colsample_bytree',
            'gamma', 'reg_alpha', 'reg_lambda', 'min_child_weight'
        ] if k in actual_params
    }

    base_strategy_params = {
        'w_trend': actual_params.get('w_trend', 0.0),
        'w_vol': actual_params.get('w_vol', 0.0),
        'w_mom': actual_params.get('w_mom', 0.0),
        'adx_th': actual_params.get('adx_th', 20.0),
        'use_slope': actual_params.get('use_slope', False),
        'proba_th': actual_params.get('proba_th', 0.5),
        'proba_th_uptrend_adj': actual_params.get('proba_th_uptrend_adj', 0.0),
        'proba_th_downtrend_adj': actual_params.get('proba_th_downtrend_adj', 0.0),
        'proba_th_sideways_adj': actual_params.get('proba_th_sideways_adj', 0.0),
        'adx_high_th': actual_params.get('adx_high_th', 25.0), # New: Load from best_params.json
        'adx_low_th': actual_params.get('adx_low_th', 20.0),   # New: Load from best_params.json
        'adx_high_proba_adj': actual_params.get('adx_high_proba_adj', -0.02), # New: Load from best_params.json
        'adx_low_proba_adj': actual_params.get('adx_low_proba_adj', 0.02),   # New: Load from best_params.json
        'downtrend_momentum_penalty_factor': actual_params.get('downtrend_momentum_penalty_factor', 1.0),
        'trend_strength_min_abs': actual_params.get('trend_strength_min_abs', 0.0),
        'proba_filter_type': actual_params.get('proba_filter_type', 'threshold'),
        'aggressive_size': actual_params.get('aggressive_size', 1.5),
        'conservative_size': actual_params.get('conservative_size', 1.0),
        'trend_strength_regime_th': actual_params.get('trend_strength_regime_th', 5.0),
        'low_trend_strength_cut_off_th': actual_params.get('low_trend_strength_cut_off_th', 0.0),
        'volatility_filter_threshold': actual_params.get('volatility_filter_threshold', 1.5),
        'atr_sl_multiplier': actual_params.get('atr_sl_multiplier', 1.2),
        'trail_ratio': actual_params.get('trail_ratio', 0.5),
        'min_hold_bars': actual_params.get('min_hold_bars', 1),

        'adx_sideways_th': actual_params.get('adx_sideways_th', 20.0), # New: Load from best_params.json
        'trend_strength_sideways_th': actual_params.get('trend_strength_sideways_th', 2.0) # New: Load from best_params.json
    }
    print("Parameters loaded successfully from best_params.json.")
except FileNotFoundError:
    print("Error: best_params.json not found. Skipping CV scenarios.")
    model_params = {}
    base_strategy_params = {}
except Exception as e:
    print(f"Error loading parameters: {e}. Skipping CV scenarios.")
    model_params = {}
    base_strategy_params = {}

# Ensure X, y, df_encoded, df are available globally from previous cells
if not all(v is not None for v in [X, y, df_encoded, df, model_params, base_strategy_params]):
    print("Error: Required variables (X, y, df_encoded, df, model_params, base_strategy_params) are not fully defined. Cannot run CV scenarios.")
else:
    # Modified scenarios to only include baseline
    scenarios_to_display = {
        "レンジ相場除外": {"filter_to_exclude": None, "exclude_sideways_regime": True, "halve_sideways_position": False}
    }

    all_scenario_results = {}
    all_scenario_trade_logs = {} # New: Dictionary to store trade logs for each scenario

    for scenario_name, params in scenarios_to_display.items():
        print(f"\n===== シナリオ: {scenario_name} ====")
        df_results_scenario, _, trade_logs_scenario = evaluate_strategy_cv(base_strategy_params, X, y, df_encoded, df, model_params,
                                                                         filter_to_exclude=params["filter_to_exclude"],
                                                                         exclude_sideways_regime=params["exclude_sideways_regime"],
                                                                         halve_sideways_position=params["halve_sideways_position"])

        # Print results for each fold as text
        print("\n各フォールドの結果 (テキスト形式):")
        for i, row in df_results_scenario.iterrows():
            print(f"--- Fold {i+1} ---")
            for col, value in row.items():
                print(f"  {col}: {value}")

        # Optionally, still print the mean for quick summary as text
        print("\nCV SUMMARY (平均 - テキスト形式):")
        mean_results = df_results_scenario.mean()
        for col, value in mean_results.items():
            print(f"  {col}: {value}")

        all_scenario_results[scenario_name] = df_results_scenario
        all_scenario_trade_logs[scenario_name] = trade_logs_scenario # New: Store trade logs

    print("\n===== 全シナリオ結果の比較 ====")
    for scenario_name, results_df in all_scenario_results.items():
        print(f"\n--- {scenario_name} ---")
        # Display key metrics as text for comparison
        mean_key_metrics = results_df[['Total Trades', 'Total Return', 'Sharpe', 'PF', 'Max DD', 'Avg_Win_PNL', 'Profit_Efficiency']].mean()
        for col, value in mean_key_metrics.items():
            print(f"  {col}: {value}")


Attempting to load parameters from best_params.json for CV scenarios...
Parameters loaded successfully from best_params.json.

===== シナリオ: レンジ相場除外 ====

各フォールドの結果 (テキスト形式):
--- Fold 1 ---
  Final Balance: 101597.83284387116
  Total Return: 1.5978328438711609
  Total Trades: 6.0
  Win Rate (%): 66.66666666666666
  PF: 1.2164483782312834
  Expectancy: 0.00012158555454596906
  Max DD: -0.021673446014048012
  Sharpe: 0.6942166191216029
  Avg_Loss: 541.377178632844
  Avg_Max_Adverse_Excursion: 0.017210673988755948
  Avg_Win_PNL: 670.1468002842114
  Profit_Efficiency: 0.7122891078413578
  Sortino: 0.0440183143132648
  Stability: 359.6770734543961
  Total Signal Days: 14.0
--- Fold 2 ---
  Final Balance: 103849.00605193812
  Total Return: 3.8490060519381197
  Total Trades: 19.0
  Win Rate (%): 47.368421052631575
  PF: 1.1914769223559694
  Expectancy: 0.0003247841640593821
  Max DD: -0.03749012779397888
  Sharpe: 0.9355906816322153
  Avg_Loss: 422.4496936351992
  Avg_Max_Adverse_Excursion: 0.0

### レンジ相場除外シナリオのCV結果

In [ ]:
if 'レンジ相場除外' in all_scenario_results:
    df_results_excluded_sideways = all_scenario_results['レンジ相場除外']
    print("\n各フォールドの結果 (レンジ相場除外 - テキスト形式):")
    for i, row in df_results_excluded_sideways.iterrows():
        print(f"--- Fold {i+1} ---")
        for col, value in row.items():
            print(f"  {col}: {value}")

    print("\nCV SUMMARY (平均 - レンジ相場除外 - テキスト形式):")
    mean_results_excluded_sideways = df_results_excluded_sideways.mean()
    for col, value in mean_results_excluded_sideways.items():
        print(f"  {col}: {value}")

    print("\n===== レンジ相場除外 シナリオの主要メトリクス (テキスト形式) ====")
    mean_key_metrics_excluded_sideways = df_results_excluded_sideways[['Total Trades', 'Total Return', 'Sharpe', 'PF', 'Max DD', 'Avg_Win_PNL', 'Profit_Efficiency']].mean()
    for col, value in mean_key_metrics_excluded_sideways.items():
        print(f"  {col}: {value}")
else:
    print("エラー: 'レンジ相場除外' シナリオの結果が見つかりません。前のセルを再実行して、すべてのシナリオが処理されたことを確認してください。")


各フォールドの結果 (レンジ相場除外 - テキスト形式):
--- Fold 1 ---
  Final Balance: 101597.83284387116
  Total Return: 1.5978328438711609
  Total Trades: 6.0
  Win Rate (%): 66.66666666666666
  PF: 1.2164483782312834
  Expectancy: 0.00012158555454596906
  Max DD: -0.021673446014048012
  Sharpe: 0.6942166191216029
  Avg_Loss: 541.377178632844
  Avg_Max_Adverse_Excursion: 0.017210673988755948
  Avg_Win_PNL: 670.1468002842114
  Profit_Efficiency: 0.7122891078413578
  Sortino: 0.0440183143132648
  Stability: 359.6770734543961
  Total Signal Days: 14.0
--- Fold 2 ---
  Final Balance: 103849.00605193812
  Total Return: 3.8490060519381197
  Total Trades: 19.0
  Win Rate (%): 47.368421052631575
  PF: 1.1914769223559694
  Expectancy: 0.0003247841640593821
  Max DD: -0.03749012779397888
  Sharpe: 0.9355906816322153
  Avg_Loss: 422.4496936351992
  Avg_Max_Adverse_Excursion: 0.019039812652280223
  Avg_Win_PNL: 897.0558875877931
  Profit_Efficiency: 0.6564891069898975
  Sortino: 0.08786393544014078
  Stability: 181.464

### ウォークフォワード検証の結果概要

In [ ]:
import json
import pandas as pd
import os

try:
    # Use the correct path to the best_params.json file defined in LUK3w6ls2Gkq
    with open(BEST_PARAMS_PATH, "r") as f:
        best_result_data = json.load(f)

    # The save_callback only saves the 'score', 'model_params', and 'strategy_params'.
    # 'fold_metrics' and 'total_trades_overall' are not stored in best_params.json.
    best_score = best_result_data.get("score")
    fold_metrics = best_result_data.get("fold_metrics")
    total_trades_overall = best_result_data.get("total_trades_overall")

    print("===== Optuna最適化のベストスコア =====")
    if best_score is not None:
        print(f"総合スコア: {best_score:.4f}")
    else:
        print("総合スコア: N/A (スコアがJSONファイルに見つかりません)")

    if fold_metrics:
        df_fold_results = pd.DataFrame(fold_metrics)
        print("\n===== ウォークフォワード各フォールドのメトリクス =====")
        display(df_fold_results)
        print("\n===== ウォークフォワードメトリクス (平均) =====")
        display(df_fold_results.mean().to_frame().T)
    else:
        print("\n注: 各フォールドのメトリクスと総取引回数はbest_params.jsonに保存されていないため表示されません。")

    if total_trades_overall is not None:
        print(f"\n全フォールドでの総取引回数: {total_trades_overall}")
    else:
        print("\n全フォールドでの総取引回数: N/A (総取引回数がJSONファイルに見つかりません)")


except FileNotFoundError:
    print(f"エラー: {BEST_PARAMS_PATH} が見つかりません。Optunaの実行を確認してください。")
except Exception as e:
    print(f"エラー発生: {e}")

===== Optuna最適化のベストスコア =====
総合スコア: 6.5367

注: 各フォールドのメトリクスと総取引回数はbest_params.jsonに保存されていないため表示されません。

全フォールドでの総取引回数: 0


In [ ]:
# import os

# # Optunaのデータベースファイルを削除
# try:
#     os.remove(DB_PATH)
#     print(f"Optunaデータベースファイル '{DB_PATH}' を削除しました。")
# except FileNotFoundError:
#     print(f"Optunaデータベースファイル '{DB_PATH}' は見つかりませんでした。")

# # best_params.json ファイルを削除
# try:
#     os.remove(BEST_PARAMS_PATH)
#     print(f"ベストパラメータファイル '{BEST_PARAMS_PATH}' を削除しました。")
# except FileNotFoundError:
#     print(f"ベストパラメータファイル '{BEST_PARAMS_PATH}' は見つかりませんでした。")

# print("Optunaの学習履歴とベストパラメータがリセットされました。再度学習を開始する場合は、関連セルを再実行してください。")